In [ ]:

(defstruct coll
            rate (term 360) (age 0) (balance 100)
          balance ; declining balance vector, element 0 is initial balance
          principal ; principal vector
          interest ; interest vector
           )

(defstruct tranche
           name
           parent_name
           split
          notional_flag
          balance ; declining balance vector, element 0 is initial balance
          principal ; principal vector
          interest ; interest vector
          )
 
(define (pay_p
      pay_from
      pay_to
      period
      )
    (let (
          (cash_avail (vector-ref (struct-ref pay_from 'principal) period))
          (remaining_balance (vector-ref (struct-ref pay_to 'balance) period))
          )
      (if (< cash_avail remaining_balance)
          (begin
              (vector-set! (struct-ref pay_from 'principal) period 0.0)
              (vector-increment! (struct-ref pay_to 'principal) period cash_avail) 
              (vector-increment! (struct-ref pay_to 'balance) period (- cash_avail))
           )
          (begin
              (vector-set! (struct-ref pay_to 'balance) period 0.0)
              (vector-increment! (struct-ref pay_to 'principal) period remaining_balance) 
              (vector-increment! (struct-ref pay_from 'principal) period (- remaining_balance))
           )
          )
      )
  )

(defstruct deal
           name
           coll_list
           (tranche_graph (make-hash-table))
            ; keys will be tranche names
            ; values will be tranche objects
            ; each tranche object will have a split
            ; object that points to its children
           root_node ; name of root of tranche graph
           closing_date
           first_pay_date
           )

(defmacro (get_tranche tranche_name)
      `(hash-table-ref tranche_graph ,tranche_name))

(defstruct (split (include: deal))
           parent_name
           child_name_list
           pay_prin_function
           )

(defstruct (seq_split (:include split))
            (pay_prin_function (lambda (period_number)
                                (do-list (b child_name_list)
                                         (pa_p (get-tranche parent) (get-tranche b) period_number))))                                   
           )


